In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt


class ActorCriticNetwork(nn.Module):
    """Actor-Critic network with shared backbone"""
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        
        # Shared layers
        self.shared_fc1 = nn.Linear(state_dim, hidden_dim)
        self.shared_fc2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Actor head (policy)
        self.actor_head = nn.Linear(hidden_dim, action_dim)
        
        # Critic head (value function)
        self.critic_head = nn.Linear(hidden_dim, 1)
        
        # Better initialization for actor head (smaller initial policy)
        nn.init.orthogonal_(self.actor_head.weight, gain=0.01)
        nn.init.constant_(self.actor_head.bias, 0)
    
    def forward(self, x):
        # Shared feature extraction
        x = F.relu(self.shared_fc1(x))
        x = F.relu(self.shared_fc2(x))
        
        # Actor output (action logits)
        logits = self.actor_head(x)
        
        # Critic output (state value)
        value = self.critic_head(x)
        
        return logits, value


def collect_trajectory(env, ac_network):
    """Collect a full episode trajectory"""
    experiences = []
    
    state, _ = env.reset()
    done = False
    
    while not done:
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        
        # Get action distribution and value estimate
        logits, value = ac_network(state_tensor)
        dist = Categorical(logits=logits)
        
        # Sample action
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        
        # Take action in environment
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        
        # Store experience
        experiences.append({
            'state': state,
            'action': action.item(),
            'reward': reward,
            'next_state': next_state,
            'done': done,
            'log_prob': log_prob,
            'entropy': entropy,
            'value': value
        })
        
        state = next_state
    
    return experiences


def compute_loss(experiences, entropy_coef=0.01, value_coef=0.5, gamma=0.99):
    """Compute Actor-Critic loss with entropy regularization"""
    
    # Calculate returns (discounted cumulative rewards)
    returns = []
    G = 0
    for exp in reversed(experiences):
        G = exp['reward'] + gamma * G
        returns.insert(0, G)
    
    # Convert to tensor (DON'T normalize returns for critic!)
    returns = torch.tensor(returns, dtype=torch.float32)
    
    # Extract values
    values = torch.cat([exp['value'] for exp in experiences])
    
    # Compute advantages (this is the key AC component!)
    advantages = returns - values.detach()  # Detach to not backprop through advantage
    
    # Normalize advantages ONLY (not returns)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # Extract log_probs and entropy
    log_probs = torch.stack([exp['log_prob'] for exp in experiences])
    entropies = torch.stack([exp['entropy'] for exp in experiences])
    
    # Compute losses (vectorized for efficiency)
    actor_loss = -(log_probs * advantages).mean()
    critic_loss = F.mse_loss(values, returns)
    entropy_loss = -entropies.mean()
    
    total_loss = actor_loss + value_coef * critic_loss + entropy_coef * entropy_loss
    
    return total_loss, actor_loss, critic_loss, entropy_loss


def train(episodes=1000, batch_size=1, lr=3e-4, entropy_coef=0.01, value_coef=0.5, gamma=0.99):
    """Train Actor-Critic policy with batching"""
    
    # Setup
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    ac_network = ActorCriticNetwork(state_dim, action_dim)
    optimizer = optim.Adam(ac_network.parameters(), lr=lr)
    
    # Tracking
    all_rewards = []
    all_losses = []
    all_actor_losses = []
    all_critic_losses = []
    all_entropies = []
    
    print("Starting Actor-Critic training...")
    print(f"Config: episodes={episodes}, batch_size={batch_size}, lr={lr}")
    print(f"        entropy_coef={entropy_coef}, value_coef={value_coef}, gamma={gamma}")
    print("-" * 70)
    
    num_updates = episodes // batch_size
    
    for update in range(num_updates):
        batch_total_losses = []
        batch_actor_losses = []
        batch_critic_losses = []
        batch_rewards = []
        batch_entropies = []
        batch_lengths = []
        
        # For accumulating advantages across batch
        all_advantages = []
        all_log_probs = []
        all_entropies_list = []
        all_values = []
        all_returns = []
        
        # Collect batch of episodes
        for _ in range(batch_size):
            experiences = collect_trajectory(env, ac_network)
            
            # Calculate returns
            returns = []
            G = 0
            for exp in reversed(experiences):
                G = exp['reward'] + gamma * G
                returns.insert(0, G)
            returns = torch.tensor(returns, dtype=torch.float32)
            
            # Extract values
            values = torch.cat([exp['value'] for exp in experiences])
            
            # Compute advantages
            advantages = returns - values.detach()
            
            # Extract log_probs and entropies
            log_probs = torch.stack([exp['log_prob'] for exp in experiences])
            entropies = torch.stack([exp['entropy'] for exp in experiences])
            
            # Accumulate for batch normalization
            all_advantages.append(advantages)
            all_log_probs.append(log_probs)
            all_entropies_list.append(entropies)
            all_values.append(values)
            all_returns.append(returns)
            
            # Track metrics
            episode_reward = sum(exp['reward'] for exp in experiences)
            episode_entropy = entropies.mean().item()
            
            batch_rewards.append(episode_reward)
            batch_entropies.append(episode_entropy)
            batch_lengths.append(len(experiences))
        
        # Concatenate all batch data
        all_advantages = torch.cat(all_advantages)
        all_log_probs = torch.cat(all_log_probs)
        all_entropies_list = torch.cat(all_entropies_list)
        all_values = torch.cat(all_values)
        all_returns = torch.cat(all_returns)
        
        # Normalize advantages across entire batch (critical!)
        all_advantages = (all_advantages - all_advantages.mean()) / (all_advantages.std() + 1e-8)
        
        # Compute losses
        actor_loss = -(all_log_probs * all_advantages).mean()
        critic_loss = F.mse_loss(all_values, all_returns)
        entropy_loss = -all_entropies_list.mean()
        
        total_loss = actor_loss + value_coef * critic_loss + entropy_coef * entropy_loss
        
        # Update network
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(ac_network.parameters(), max_norm=0.5)
        optimizer.step()
        
        # Store metrics
        avg_reward = np.mean(batch_rewards)
        avg_entropy = np.mean(batch_entropies)
        avg_length = np.mean(batch_lengths)
        
        all_rewards.append(avg_reward)
        all_losses.append(total_loss.item())
        all_actor_losses.append(actor_loss.item())
        all_critic_losses.append(critic_loss.item())
        all_entropies.append(avg_entropy)
        
        # Logging
        if update % 5 == 0:
            print(f"Update {update:3d} | "
                  f"Reward: {avg_reward:6.1f} | "
                  f"Length: {avg_length:5.1f} | "
                  f"A_Loss: {actor_loss.item():6.3f} | "
                  f"C_Loss: {critic_loss.item():6.3f} | "
                  f"Entropy: {avg_entropy:.3f}")
    
    env.close()
    
    print("-" * 70)
    print("Training complete!")
    print(f"Final average reward: {all_rewards[-1]:.1f}")
    
    return ac_network, all_rewards, all_losses, all_actor_losses, all_critic_losses, all_entropies


def plot_training(rewards, losses, actor_losses, critic_losses, entropies):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # Rewards
    axes[0, 0].plot(rewards, color='green', linewidth=2)
    axes[0, 0].set_xlabel('Update')
    axes[0, 0].set_ylabel('Average Reward')
    axes[0, 0].set_title('Training Rewards')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Total Loss
    axes[0, 1].plot(losses, color='red', linewidth=2)
    axes[0, 1].set_xlabel('Update')
    axes[0, 1].set_ylabel('Total Loss')
    axes[0, 1].set_title('Total Loss')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Entropy
    axes[0, 2].plot(entropies, color='purple', linewidth=2)
    axes[0, 2].set_xlabel('Update')
    axes[0, 2].set_ylabel('Entropy')
    axes[0, 2].set_title('Policy Entropy')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Actor Loss
    axes[1, 0].plot(actor_losses, color='blue', linewidth=2)
    axes[1, 0].set_xlabel('Update')
    axes[1, 0].set_ylabel('Actor Loss')
    axes[1, 0].set_title('Actor Loss (Policy)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Critic Loss
    axes[1, 1].plot(critic_losses, color='orange', linewidth=2)
    axes[1, 1].set_xlabel('Update')
    axes[1, 1].set_ylabel('Critic Loss')
    axes[1, 1].set_title('Critic Loss (Value)')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Combined view
    axes[1, 2].plot(actor_losses, label='Actor', alpha=0.7, linewidth=2)
    axes[1, 2].plot(critic_losses, label='Critic', alpha=0.7, linewidth=2)
    axes[1, 2].set_xlabel('Update')
    axes[1, 2].set_ylabel('Loss')
    axes[1, 2].set_title('Actor vs Critic Loss')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ac_training_curves.png', dpi=150, bbox_inches='tight')
    print("Saved training curves to 'ac_training_curves.png'")
    plt.show()


def evaluate_policy(ac_network, num_episodes=10, render=False):
    """Evaluate trained Actor-Critic policy"""
    if render:
        env = gym.make('CartPole-v1', render_mode='human')
    else:
        env = gym.make('CartPole-v1')
    
    total_rewards = []
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        episode_reward = 0
        
        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            
            with torch.no_grad():
                logits, value = ac_network(state_tensor)
                dist = Categorical(logits=logits)
                action = dist.sample()
            
            state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            episode_reward += reward
        
        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1}: Reward = {episode_reward}")
    
    env.close()
    
    avg_reward = np.mean(total_rewards)
    std_reward = np.std(total_rewards)
    print(f"\nEvaluation Results:")
    print(f"Average Reward: {avg_reward:.1f} ± {std_reward:.1f}")
    
    return total_rewards


if __name__ == "__main__":
    # Train Actor-Critic
    ac_network, rewards, losses, actor_losses, critic_losses, entropies = train(
        episodes=1000,
        batch_size=10,  # Batch across 10 episodes
        lr=3e-4,
        entropy_coef=0.01,
        value_coef=0.5,
        gamma=0.99
    )
    
    # Plot results
    plot_training(rewards, losses, actor_losses, critic_losses, entropies)
    
    # Evaluate
    print("\n" + "=" * 70)
    print("Evaluating trained Actor-Critic policy...")
    print("=" * 70)
    evaluate_policy(ac_network, num_episodes=20)

Starting Actor-Critic training...
Config: episodes=1000, batch_size=10, lr=0.0003
        entropy_coef=0.01, value_coef=0.5, gamma=0.99
----------------------------------------------------------------------


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 11 but got size 40 for tensor number 1 in the list.